# BRmoral — avaliação de posição perante temas

Notebook complementar ao **TCC.ipynb**, seguindo sua sequência de instalação,
tipos/variáveis, métodos, testes e comparação. Resultados da rodada de 17/09/2026.

**Pergunta:** os embeddings E5 permitem classificar posição melhor quando recebem
uma cabeça supervisionada? A comparação entre encoder congelado e contrastivo
continua incompleta; somente os baselines foram executados.

## O que é o BRmoral?

É um corpus de opiniões em português brasileiro produzidas por participantes que
também declararam sua posição perante cada tema. A versão obtida tem **510 autores
e oito textos por autor**, totalizando 4.080 opiniões potenciais. Os temas são:
casamento entre pessoas do mesmo sexo, porte de armas, aborto, pena de morte,
legalização das drogas, redução da maioridade penal, cotas raciais e isenção de
impostos para igrejas. Também há dados de perfil e fundamentos morais no original;
eles não são usados como entradas ou rótulos neste experimento.

Cada exemplo é **(texto, tema, posição)**. O mesmo assunto pode aparecer em textos
favoráveis ou contrários: similaridade temática não basta para reconhecer posição.
Aqui não convertemos a ideologia autodeclarada de uma pessoa em rótulo de seus textos.

Fonte acadêmica: Silva e Paraboni (2023), *Politically-oriented information inference
from text*, §§3 e 4.1.2, pp.575–578, arquivo
`artigos/Politically-oriented information inference from text.pdf`.
Descrição dos campos e licença: [README original extraído](analises/brmoral_dataset_readme.txt).
Distribuição v5.10, setembro de 2019, **CC BY 4.0**; não confundir a versão do
arquivo com o ano de publicação dos artigos que o utilizam.

## Instalação e Imports

Para ler e recalcular tabelas: Python e NumPy do ambiente do projeto. Para repetir
modelos, use `.venv-gpu` com `investigacao/requirements-gpu.txt`.
Não há instalação automática nem download ao executar todas as células.

Rastreabilidade **R4**: Kapoor e Narayanan (2023), *Leakage and the reproducibility
crisis in machine-learning-based science*, DOI 10.1016/j.patter.2023.100804.
Adaptação: caminhos explícitos, manifestos e hashes; não reproduzimos um experimento
desse artigo. Execute este notebook a partir da raiz do projeto.


In [1]:
import json
import hashlib
import subprocess
import sys
from pathlib import Path
from collections import Counter
from typing import TypedDict
from investigacao.protocolo import validate_corpus, classification_metrics, digest

ROOT = Path.cwd()
assert (ROOT / 'TCC.ipynb').is_file(), 'Abra o notebook na raiz do projeto.'
print('Python atual:', sys.version.split()[0])
print('Raiz:', ROOT)


Python atual: 3.12.14
Raiz: C:\Users\User\Desktop\Micael\code\TCC


## Tipos e Variáveis
### Tipos

**R3/R9:** Silva e Paraboni (2023), §§3–4.1.2, e README v5.10. O esquema
abaixo distingue posição por texto/tema, autoria e partição. Tipagem é uma
adaptação de engenharia; a auditoria posterior verifica os valores reais.


In [2]:
class Exemplo(TypedDict):
    """R3/R9: posição por tema; autoria é usada apenas no agrupamento."""
    id: str
    text: str
    target: str
    label: str
    author_id: str
    family_id: str
    split: str
    label_source: str

print('Campos:', ', '.join(Exemplo.__annotations__))


Campos: id, text, target, label, author_id, family_id, split, label_source


### Variáveis
#### Modelos e execuções

**R4/R6/R7:** configurações congeladas da execução, descritas em
[METODOLOGIA.md](investigacao/METODOLOGIA.md). E5 gera embeddings; regressão
logística aprende a fronteira; NLI compara o texto com uma hipótese de apoio.
As tabelas leem artefatos reais, sem executar novamente o treinamento.


In [3]:
DATA = ROOT / 'analises/dados/brmoral_20260917'
RUN = ROOT / 'analises/execucoes/brmoral_baselines_gpu_20260917'
DOC = ROOT / 'analises/execucoes/documento_6x1_brmoral_gpu_20260917'

def load(path):
    """R4: leitura explícita de artefato UTF-8 persistido."""
    return json.loads(path.read_text(encoding='utf-8'))

rows: list[Exemplo] = [json.loads(s) for s in (DATA / 'corpus.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
meta = load(DATA / 'corpus.meta.json')
metrics = load(RUN / 'metricas.json')
predictions = load(RUN / 'previsoes.json')
selection = load(RUN / 'frozen_selection.json')
manifest = load(RUN / 'manifest.json')
print('Corpus:', meta['name'], meta['version'], meta['license'])
print('GPU da rodada:', manifest['gpu']['name'])
print('Seleção completa:', selection['comparison_complete'])


Corpus: BRmoral 5.10-sept2019 CC-BY-4.0
GPU da rodada: NVIDIA GeForce GTX 1060 3GB
Seleção completa: False


### Métodos Gerais

**R4:** tabelas calculadas diretamente dos artefatos, com arredondamento somente
na apresentação. A função abaixo formata saídas legíveis sem bibliotecas extras.


In [4]:
def table(headers, values):
    """R4: formatação; não altera dados ou métricas."""
    values = [[str(v) for v in row] for row in values]
    widths = [max(len(str(h)), *(len(r[i]) for r in values)) for i, h in enumerate(headers)]
    print(' | '.join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print('-+-'.join('-' * w for w in widths))
    for row in values:
        print(' | '.join(v.ljust(w) for v, w in zip(row, widths)))

print('Métodos auxiliares disponíveis.')


Métodos auxiliares disponíveis.


## Testes
### A0 — Dados, exclusões e partições

**R9:** Santos e Paraboni (2019), *Moral Stance Recognition and Polarity
Classification from Twitter and Elicited Text*, §3.1, p.1071,
DOI 10.26615/978-954-452-056-4_123; README v5.10.
Escores 0/1 → contra; 2/3 → neutro; 4/5 → favor. A tarefa atual é binária:
neutros e registros sem texto/escore são excluídos, nunca convertidos em contra.

**Adaptação R4:** 64/16/20% dos componentes autor/duplicata, seed 42.
Não é uma divisão oficial do corpus. Autores e duplicatas não cruzam partições;
os oito temas permanecem conhecidos. Isso testa novos autores, não novos temas.


In [5]:
audit = validate_corpus(rows, meta)
assert audit['sha256'] == load(DATA / 'audit.json')['sha256']
archive = ROOT / 'analises/brmoral_original.download'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == meta['archive_sha256']
for path_string, expected in manifest['input_hashes'].items():
    path = ROOT / Path(path_string.replace('\\', '/'))
    assert digest(path.read_text(encoding='utf-8')) == expected
print('Auditoria e hashes: OK')
print('Exclusões:', meta['exclusions'])
table(['Partição', 'Exemplos', 'Autores', 'Componentes', 'Contra', 'Favor'], [
    [s, sum(r['split'] == s for r in rows),
     len({r['author_id'] for r in rows if r['split'] == s}),
     len({r['family_id'] for r in rows if r['split'] == s}),
     sum(r['split'] == s and r['label'] == 'against' for r in rows),
     sum(r['split'] == s and r['label'] == 'favor' for r in rows)]
    for s in ['train', 'dev', 'test']])


Auditoria e hashes: OK
Exclusões: {'neutral': 861, 'missing_text_or_score': 8}
Partição | Exemplos | Autores | Componentes | Contra | Favor
---------+----------+---------+-------------+--------+------
train    | 2060     | 325     | 325         | 963    | 1097 
dev      | 510      | 81      | 81          | 242    | 268  
test     | 641      | 102     | 102         | 295    | 346  


### A1 — Exemplo de entrada

**R3/R9:** inspecionamos somente um exemplo de treino para explicar o contrato.
`gun-control` é o nome histórico da coluna, mas seu alvo corresponde ao porte de
armas; a tradução literal não deve inverter as classes. Não mostramos perfil
demográfico e não o fornecemos ao modelo.


In [6]:
example = next(r for r in rows if r['split'] == 'train')
print('Tema:', example['target'])
print('Posição:', example['label'])
print('Texto:', example['text'])


Tema: casamento entre pessoas do mesmo sexo
Posição: favor
Texto: Acredito que todos possuem direito de escolher a forma como querem viver ou buscar a felicidade. Essa escolha deve ser reconhecida como legítima e legal.


### B0 — E5 sem ajuste

**R6:** Tunstall et al. (2022), *Efficient Few-Shot Learning Without Prompts*,
§3.1, https://arxiv.org/abs/2209.11055, contextualiza classificadores sobre
embeddings. Nosso baseline de templates é uma adaptação própria, não uma
reprodução do SetFit: codifica `Alvo: ... Texto: ...` e compara com frases fixas
de apoio/oposição, usando margem de cossenos e limiar zero. Prefixo `query:`,
embeddings normalizados, máximo 512 tokens. Ver `modelos.zero_shot`.

Macro-F1 é a média do F1 das duas classes; acurácia balanceada é a média de seus
recalls; MCC mede associação entre previsões e rótulos. **R3/R4** fundamentam a
avaliação separada e suas limitações. Macro-F1 global não é média entre temas.


In [7]:
global_metrics = [r for r in metrics if 'target' not in r]
baseline = next(r for r in global_metrics if r['method'] == 'sem_ajuste')
print(json.dumps(baseline, ensure_ascii=False, indent=2))


{
  "method": "sem_ajuste",
  "split": "test",
  "n": 641,
  "labels": [
    "against",
    "favor"
  ],
  "confusion": [
    [
      52,
      243
    ],
    [
      19,
      327
    ]
  ],
  "macro_f1": 0.49906340229555923,
  "balanced_accuracy": 0.5606789458214951,
  "mcc": 0.19273051327761,
  "missing_classes": []
}


### B1 — E5 congelado + regressão logística

**R6**, §3.1: separação entre representação e cabeça classificadora. Adaptação:
E5 permanece congelado; cabeça logística L2, pesos balanceados, grade
C={0,01; 0,1; 1; 10}. Treino ajusta coeficientes; dev escolhe C por macro-F1.
Embeddings são calculados em CUDA; a cabeça é treinada na CPU.
As seeds 13/42/77 não alteram as partições e podem produzir resultados idênticos.


In [8]:
table(['Seed', 'C escolhido', 'Macro-F1 dev', 'Macro-F1 teste'], [
    [seed, load(RUN / f'congelado_{seed}/selection.json')['C'],
     f"{load(RUN / f'congelado_{seed}/selection.json')['dev']['macro_f1']:.4f}",
     f"{next(r for r in global_metrics if r['method'] == 'congelado' and r['seed'] == seed)['macro_f1']:.4f}"]
    for seed in [13, 42, 77]])
print('Escolha por dev:', selection['chosen_method'])


Seed | C escolhido | Macro-F1 dev | Macro-F1 teste
-----+-------------+--------------+---------------
13   | 10.0        | 0.8349       | 0.8436        
42   | 10.0        | 0.8349       | 0.8436        
77   | 10.0        | 0.8349       | 0.8436        
Escolha por dev: congelado


### C1 — NLI relacional

**R7:** Yin, Hay e Roth (2019), *Benchmarking Zero-shot Text Classification:
Datasets, Evaluation and Entailment Approach*, §3, https://aclanthology.org/D19-1404/.
Adaptação: texto como premissa, “O autor é favorável a {alvo}.” como hipótese.
Comparamos entailment e contradiction para uma previsão binária. A probabilidade
neutral não significa neutralidade política. NLI é comparador, não participa
da escolha entre B0 e B1 no desenvolvimento.


In [9]:
nli = next(r for r in global_metrics if r['method'] == 'nli_relacional')
print(json.dumps(nli, ensure_ascii=False, indent=2))


{
  "method": "nli_relacional",
  "split": "test",
  "n": 641,
  "labels": [
    "against",
    "favor"
  ],
  "confusion": [
    [
      209,
      86
    ],
    [
      133,
      213
    ]
  ],
  "macro_f1": 0.6583330290971487,
  "balanced_accuracy": 0.6620407563436858,
  "mcc": 0.3237834658172986,
  "missing_classes": []
}


### Comparação — resultados externos

**R3/R4:** mesmo teste e rótulos para todos os métodos. A célula recalcula as
matrizes a partir das previsões para verificar consistência com o relatório.
Não escolhe parâmetros pelo teste. Consulte `METODOLOGIA.md` para referências
completas e diferenças em relação aos trabalhos originais.


In [10]:
for metric in metrics:
    subset = [r for r in predictions if r['method'] == metric['method']
              and r.get('seed', -1) == metric.get('seed', -1)
              and ('target' not in metric or r['target'] == metric['target'])]
    observed = classification_metrics([r['truth'] for r in subset], [r['prediction'] for r in subset])
    assert observed['confusion'] == metric['confusion']
summary = [r for r in global_metrics if r.get('seed', 13) == 13]
table(['Método', 'N', 'Macro-F1', 'BA', 'MCC'], [
    [r['method'], r['n'], *[f"{r[k]:.4f}" for k in ['macro_f1', 'balanced_accuracy', 'mcc']]] for r in summary])
print('Matrizes conferidas:', len(metrics))


Método         | N   | Macro-F1 | BA     | MCC   
---------------+-----+----------+--------+-------
sem_ajuste     | 641 | 0.4991   | 0.5607 | 0.1927
congelado      | 641 | 0.8436   | 0.8455 | 0.6889
nli_relacional | 641 | 0.6583   | 0.6620 | 0.3238
Matrizes conferidas: 45


### B2 — Incerteza do ganho

**R8:** Dror et al. (2018), *The Hitchhiker’s Guide to Testing Statistical
Significance in Natural Language Processing*, https://aclanthology.org/P18-1128/.
Adaptação com **R4**: bootstrap pareado por componentes de autor/duplicata do
teste, 1.000 repetições; intervalo percentil de 95% da diferença de macro-F1.
É condicional ao split e treinamento observados, sem estimar variação entre treinos.


In [11]:
interval = load(RUN / 'intervalo_baselines.json')
print(json.dumps(interval, ensure_ascii=False, indent=2))


{
  "comparison": "congelado_menos_sem_ajuste",
  "seed": 13,
  "delta_macro_f1": 0.3445646226849247,
  "percentile_interval_95": [
    0.2993947233115392,
    0.39035290499098724
  ],
  "groups": 102,
  "repeats": 1000,
  "method": "paired_cluster_percentile_bootstrap"
}


### B3 — Error Analysis / resultados por tema

**R3/R4:** decompor resultados evita atribuir o desempenho agregado a todos os
temas. A matriz usa linhas verdadeiras e colunas previstas, ordem contra/favor.
Esta análise posterior ao teste serve para diagnóstico, não para retunar a rodada.


In [12]:
by_topic = [r for r in metrics if 'target' in r and r['method'] == 'congelado' and r.get('seed') == 13]
table(['Tema', 'N', 'Macro-F1', 'Acertos favor / total favor'], [
    [r['target'], r['n'], f"{r['macro_f1']:.4f}",
     f"{r['confusion'][1][1]} / {sum(r['confusion'][1])}"] for r in by_topic])


Tema                                  | N  | Macro-F1 | Acertos favor / total favor
--------------------------------------+----+----------+----------------------------
casamento entre pessoas do mesmo sexo | 95 | 0.8279   | 91 / 91                    
cotas raciais                         | 79 | 0.6644   | 32 / 50                    
isenção de impostos para igrejas      | 82 | 0.4533   | 0 / 13                     
legalização das drogas                | 77 | 0.7966   | 54 / 56                    
legalização do aborto                 | 79 | 0.6111   | 63 / 67                    
legalização do porte de armas         | 76 | 0.7816   | 12 / 22                    
pena de morte                         | 76 | 0.6961   | 5 / 15                     
redução da maioridade penal           | 77 | 0.8556   | 29 / 32                    


**Interpretação:** a cabeça supervisionada melhora muito o resultado global,
mas falha na classe favorável à isenção de impostos para igrejas (0/13 acertos).
Casamento tem somente quatro exemplos contra no teste; estimativas por classe
ficam frágeis. O agregado 0,844 não representa qualidade uniforme por tema.

### D0 — Aplicação ao documento 6×1

**R3/R7:** transferência de domínio, não avaliação com gabarito. A recuperação
usa trechos e perguntas; a cabeça recebe a pergunta como alvo, diferentemente dos
alvos nominais do corpus. Limiar exploratório 0,7, sem calibração documental.
Probabilidade não é intensidade; `response` continua ausente.


In [13]:
transfer = load(DOC / 'transferencia_externa.json')
assert len(transfer['items']) == 70
assert all(r['response'] is None for r in transfer['items'])
assert transfer['selection']['comparison_complete'] is False
counts = Counter(r['state'] for r in transfer['items'])
table(['Estado', 'Perguntas'], sorted(counts.items()))
print('Transferência exploratória:', transfer['transfer_exploratory'])


Estado    | Perguntas
----------+----------
contraria | 2        
favoravel | 64       
incerta   | 4        
Transferência exploratória: True


**Interpretação:** 64 favoráveis, 2 contrárias e 4 incertas não comprovam acerto.
A concentração de favoráveis pode refletir comportamento fora do domínio;
não temos gabarito para resolver essa dúvida. Não produzimos escore político validado.

### Custo e limite de hardware

**R4:** manifesto mede esta execução, não um benchmark isolado de GPU. A estimativa
de treinamento float32/AdamW inclui pesos, gradientes e dois estados; exclui
ativações e temporários. É uma verificação de engenharia, não resultado científico.


In [14]:
memory = load(ROOT / 'analises/brmoral_training_memory.json')
print('Tempo externo (s):', round(manifest['elapsed_seconds'], 2))
print('Pico CUDA alocado (GiB):', round(manifest['gpu']['peak_allocated_bytes'] / 2**30, 3))
print('Estimativa mínima treinamento (GB decimais):', round(memory['float32_weights_grad_adam_two_moments_bytes'] / 1e9, 2))
print('Pendente:', manifest['config']['pending'])


Tempo externo (s): 110.08
Pico CUDA alocado (GiB): 1.148
Estimativa mínima treinamento (GB decimais): 4.45
Pendente: ['contrastivo']


### Reprodução opcional na GPU

**R4/R6/R7/R9:** orquestração dos métodos já descritos. Por padrão, apenas mostra
o comando; altere `REEXECUTAR_GPU` para repetir os baselines em uma nova pasta.
O comando exige CUDA e não faz fallback silencioso. Não executa ajuste contrastivo.


In [15]:
REEXECUTAR_GPU = False
PYTHON_GPU = ROOT / '.venv-gpu/Scripts/python.exe'
NEW_RUN = ROOT / 'analises/execucoes/brmoral_notebook_nova_rodada'
command = [str(PYTHON_GPU), '-m', 'investigacao', 'external',
           '--corpus', str(DATA / 'corpus.jsonl'), '--metadata', str(DATA / 'corpus.meta.json'),
           '--output', str(NEW_RUN), '--device', 'cuda', '--batch-size', '1',
           '--low-vram', '--baselines-only']
print(subprocess.list2cmdline(command))
if REEXECUTAR_GPU:
    assert PYTHON_GPU.is_file(), 'Ambiente GPU não encontrado.'
    assert not NEW_RUN.exists(), 'Escolha uma pasta nova para preservar as execuções.'
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Modo leitura: nenhum modelo foi reexecutado nesta sessão.')


C:\Users\User\Desktop\Micael\code\TCC\.venv-gpu\Scripts\python.exe -m investigacao external --corpus C:\Users\User\Desktop\Micael\code\TCC\analises\dados\brmoral_20260917\corpus.jsonl --metadata C:\Users\User\Desktop\Micael\code\TCC\analises\dados\brmoral_20260917\corpus.meta.json --output C:\Users\User\Desktop\Micael\code\TCC\analises\execucoes\brmoral_notebook_nova_rodada --device cuda --batch-size 1 --low-vram --baselines-only
Modo leitura: nenhum modelo foi reexecutado nesta sessão.


### D1 — Diagnóstico de transferência ao 8values

**R5/R7:** CheckList (Ribeiro et al., 2020, §2) e NLI (Yin et al., 2019, §3).
280 textos construídos: apoio/oposição em duas formas para cada uma das 70
perguntas. O texto da proposição é preservado; `effect` não fornece os rótulos.
Modelos e limiares congelados antes da execução em GPU. As expectativas são
por construção, não anotações humanas ou estimativas de acurácia documental.

Macro-F1 abaixo força uma escolha binária. Cobertura e acerto condicional usam
o limiar 0,7 da cabeça; NLI também exige margem 0,2. Ordenação verifica apenas
se o apoio recebe escore maior; ambos corretos exige decisões absolutas certas.


In [16]:
TRANSFER_RUN = ROOT / 'analises/execucoes/transferencia_8values_gpu_20260917'
if (TRANSFER_RUN / 'metrics.json').exists():
    transfer_metrics = load(TRANSFER_RUN / 'metrics.json')
    table(['Método', 'Macro-F1', 'Cobertura', 'Acerto aceitos', 'Ordenação', 'Ambos corretos'], [
        [r['method'], *[f"{r[k]:.4f}" if r[k] is not None else 'n/a' for k in
         ['macro_f1', 'direction_coverage', 'accuracy_when_accepted', 'pair_ordering', 'both_sides_correct']]]
        for r in transfer_metrics if r['template'] == 'all'])
else:
    print('Diagnóstico ainda não disponível nesta cópia do projeto.')


Método            | Macro-F1 | Cobertura | Acerto aceitos | Ordenação | Ambos corretos
------------------+----------+-----------+----------------+-----------+---------------
brmoral_congelado | 0.7785   | 0.5179    | 0.8828         | 1.0000    | 0.5571        
nli_pergunta      | 0.3333   | 1.0000    | 0.5000         | 0.3643    | 0.0000        


**Conclusão do diagnóstico:** cabeça BRmoral: macro-F1 0,779 e ordenação
100%, porém ambos os lados corretos em apenas 55,7% dos pares. Com confiança
0,7, decide em 51,8% dos casos e acerta 88,3% dessas decisões; ainda há 17
erros entre 145 decisões aceitas. Não calibramos limiares nesses resultados.
NLI com hipótese literal da pergunta classificou todos os 280 casos como apoio,
falhando nas 140 rejeições. Isso demonstra uma falha nestes templates citacionais,
não que NLI falhe em todo texto natural. Não há liberação de pontuação 8values.
Ver [relatório da transferência](analises/transferencia_8values_20260917.md).

## Conclusões

1. O BRmoral permitiu avaliação externa real, com posição por texto/tema e
   separação de autores/duplicatas; 3.211 exemplos binários após exclusões.
2. E5 congelado + regressão logística alcançou macro-F1 **0,844**, contra **0,499**
   sem ajuste e **0,658** do NLI. Melhorar a fronteira de decisão ajudou neste corpus.
3. O ganho não é uniforme entre temas e não valida automaticamente documentos
   longos, perguntas 8values ou pontuações de intensidade.
4. A transferência para o documento 6×1 permanece exploratória e sem gabarito.
5. Ainda falta testar ajuste contrastivo do mesmo encoder; float32/AdamW completo
   excede os 3 GB da GPU. Não é possível concluir se adaptar a representação ajuda.

## Referências e rastreabilidade

- Silva e Paraboni (2023), *Politically-oriented information inference from text*,
  §§3–4.1.2, [artigo local](artigos/Politically-oriented%20information%20inference%20from%20text.pdf).
- Santos e Paraboni (2019), *Moral Stance Recognition and Polarity Classification
  from Twitter and Elicited Text*, §3.1, DOI 10.26615/978-954-452-056-4_123.
- Pavan et al., *Morality Classification in Natural Language Text*,
  DOI 10.1109/TAFFC.2020.3034050: referência solicitada pela distribuição original.
  O README cita 2020; Silva e Paraboni citam a publicação em 2023.
- Kapoor e Narayanan (2023), Tunstall et al. (2022), Yin et al. (2019) e
  Dror et al. (2018): identificados junto das etapas correspondentes.
- [Métodos, referências completas e adaptações](investigacao/METODOLOGIA.md).
- [Relatório da rodada](analises/resultados_brmoral_gpu_20260917.md).

**Validação:** a rodada de implementação passou em 16 testes. Este notebook
adiciona auditoria dos dados, hashes, recálculo das matrizes e verificações dos
70 resultados documentais. Suas células foram executadas em sequência na geração;
os modelos são resultados persistidos da execução GPU, não novo treinamento.


## D2 — Ablação de formato e hipótese NLI

**R5/R7:** CheckList (Ribeiro et al., 2020, §2) e Yin et al. (2019, §3).
Fatorial aspas × posição da concordância/discordância: 560 casos em 70 famílias.
A proposição permanece literal mesmo sem aspas; não são paráfrases ou documentos
naturais. A segunda hipótese NLI trata da concordância do autor, não da verdade
da proposição. Comparação exploratória pré-definida; sem treino ou calibração.

Usamos a rodada v2, que preserva a regência “discordo da/dessa” do diagnóstico
anterior. A primeira rodada está preservada, mas não é a comparação principal.


In [17]:
FORMAT_RUN = ROOT / 'analises/execucoes/ablacao_formato_8values_gpu_20260917_v2'
if (FORMAT_RUN / 'metrics.json').exists():
    format_metrics = load(FORMAT_RUN / 'metrics.json')
    table(['Método', 'Formato', 'Macro-F1', 'Cobertura'], [
        [r['method'], r['template'], f"{r['macro_f1']:.4f}", f"{r['direction_coverage']:.4f}"]
        for r in format_metrics])
else:
    print('Rodada de formato ainda não disponível nesta cópia.')


Método            | Formato          | Macro-F1 | Cobertura
------------------+------------------+----------+----------
brmoral_congelado | all              | 0.7875   | 0.5625   
brmoral_congelado | aspas_antes      | 0.7786   | 0.5286   
brmoral_congelado | aspas_depois     | 0.8212   | 0.6429   
brmoral_congelado | sem_aspas_antes  | 0.7272   | 0.4714   
brmoral_congelado | sem_aspas_depois | 0.8214   | 0.6071   
nli_pergunta      | all              | 0.3681   | 0.9696   
nli_pergunta      | aspas_antes      | 0.3333   | 1.0000   
nli_pergunta      | aspas_depois     | 0.4081   | 0.9000   
nli_pergunta      | sem_aspas_antes  | 0.3333   | 1.0000   
nli_pergunta      | sem_aspas_depois | 0.3939   | 0.9786   
nli_posicao_autor | all              | 0.9803   | 0.9768   
nli_posicao_autor | aspas_antes      | 0.9857   | 0.9786   
nli_posicao_autor | aspas_depois     | 0.9499   | 0.9357   
nli_posicao_autor | sem_aspas_antes  | 0.9929   | 0.9929   
nli_posicao_autor | sem_aspas_depois | 0

**Limite:** melhoria em templates explícitos não demonstra validade
documental. Não promovemos a nova hipótese a configuração final nem calculamos
pontuações 8values. Consulte o [relatório](analises/ablacao_formato_8values_20260917.md)
para resultados e próximos testes em linguagem natural.


## D3 — Textos naturais com temas reservados

**R3/R4/R6/R7/R8:** unidade texto-alvo, controle de vazamento, cabeça congelada,
hipóteses NLI e comparação pareada; referências em METODOLOGIA.md.
Em cada uma de oito rodadas, retiramos um tema de treino e dev. O teste contém
esse tema e autores que não participaram do ajuste. C usa apenas dev dos demais
temas. O agregado da cabeça reúne oito modelos, não um único classificador.
NLI compara proposição normativa versus concordância do autor, sem treinamento.

O teste já foi consultado anteriormente: avaliação exploratória, não novo teste
cego. Os rótulos são naturais do corpus (neutros excluídos), não templates.


In [18]:
TOPIC_RUN = ROOT / 'analises/execucoes/temas_reservados_gpu_20260917'
if (TOPIC_RUN / 'metrics.json').exists():
    topic_metrics = load(TOPIC_RUN / 'metrics.json')
    table(['Método', 'Macro-F1', 'BA', 'Cobertura', 'Acerto aceitos'], [
        [r['method'], *[f"{r[k]:.4f}" if r[k] is not None else 'n/a' for k in
         ['macro_f1', 'balanced_accuracy', 'coverage', 'accepted_accuracy']]]
        for r in topic_metrics if r['target'] is None])
    table(['Tema', 'Método', 'N', 'Macro-F1'], [
        [r['target'], r['method'], r['n'], f"{r['macro_f1']:.4f}"]
        for r in topic_metrics if r['target'] is not None and r['method'] != 'maioria_treino'])
    print('Diferença entre hipóteses NLI:', load(TOPIC_RUN / 'interval_nli.json'))
else:
    print('Rodada de temas reservados ainda não disponível.')


Método            | Macro-F1 | BA     | Cobertura | Acerto aceitos
------------------+----------+--------+-----------+---------------
e5_tema_reservado | 0.5230   | 0.5235 | 0.4743    | 0.5461        
maioria_treino    | 0.2654   | 0.2988 | 1.0000    | 0.3183        
nli_posicao_autor | 0.7036   | 0.7084 | 0.5304    | 0.7382        
nli_proposicao    | 0.6905   | 0.6903 | 0.5959    | 0.7592        
Tema                                  | Método            | N  | Macro-F1
--------------------------------------+-------------------+----+---------
casamento entre pessoas do mesmo sexo | e5_tema_reservado | 95 | 0.2620  
cotas raciais                         | e5_tema_reservado | 79 | 0.4726  
isenção de impostos para igrejas      | e5_tema_reservado | 82 | 0.1431  
legalização das drogas                | e5_tema_reservado | 77 | 0.7075  
legalização do aborto                 | e5_tema_reservado | 79 | 0.5756  
legalização do porte de armas         | e5_tema_reservado | 76 | 0.2842  
pena d

**Resultado:** E5 com tema reservado obteve macro-F1 0,523; NLI literal
0,690 e NLI sobre concordância do autor 0,704. A diferença entre hipóteses NLI
foi +0,013, com IC95% [-0,022; 0,050]: não há vantagem clara nesta rodada.
A hipótese sobre o autor piorou em cinco dos oito temas. O sucesso nos templates
não se repetiu com a mesma magnitude nos textos naturais.

A avaliação permite confrontar o ganho em templates com textos naturais,
mas não valida escala de intensidade 8values. Hipóteses e limiares não são
escolhidos pelos resultados desta rodada. Consulte o
[relatório](analises/temas_reservados_20260917.md).


## D4 — Seleção de decisão e abstenção no desenvolvimento

**R4/R7:** isolamento dos dados e hipóteses NLI; referências completas em
METODOLOGIA.md. Ajustamos limiares, não probabilidades. Excluímos do dev o tema
avaliado e escolhemos a maior cobertura com acerto empírico ≥80%, cobertura ≥25%
e dez aceitos por classe. Sem opção elegível, abstenção total. Requisitos locais
pré-especificados, sem garantia para outro tema. Seleções congeladas antes de
carregar previsões de teste; teste já consultado, avaliação exploratória.


In [19]:
SELECTIVE_RUN = ROOT / 'analises/execucoes/calibracao_seletiva_gpu_20260917'
if (SELECTIVE_RUN / 'metrics.json').exists():
    selective_results = load(SELECTIVE_RUN / 'metrics.json')
    table(['Política', 'Aceitos', 'Acertos', 'Erros', 'Cobertura', 'Acerto aceitos'], [
        [r['policy'], r['accepted'], r['correct'], r['errors'],
         f"{r['coverage']:.4f}", f"{r['accepted_accuracy']:.4f}" if r['accepted_accuracy'] is not None else 'n/a']
        for r in selective_results if r['target'] is None])
else:
    print('Seleção de limiares ainda não disponível nesta cópia.')


Política               | Aceitos | Acertos | Erros | Cobertura | Acerto aceitos
-----------------------+---------+---------+-------+-----------+---------------
nli_posicao_autor_fixo | 340     | 251     | 89    | 0.5304    | 0.7382        
nli_proposicao_fixo    | 382     | 290     | 92    | 0.5959    | 0.7592        
selecionada_dev        | 274     | 192     | 82    | 0.4275    | 0.7007        


**Conclusão:** a seleção aceitou 274/641 casos e acertou 192 (70,1% dos
aceitos), abaixo do NLI literal fixo, que aceitou 382 e acertou 290 (75,9%).
Menos erros absolutos vieram acompanhados de menor cobertura e menor acerto
condicional. Para pena de morte, nenhum candidato cumpriu a meta no dev e houve
abstenção total. Não promovemos essa política ao pipeline documental.
Ver [relatório](analises/calibracao_seletiva_20260917.md). Suite: 21 testes aprovados.
